# CatBoost v2: подбор регуляризации

Этот ноутбук продолжает эксперимент с `depth=6`, выбранным в `07_catboost_tuning.ipynb`. Набор `enhanced_v2` из 216 признаков, временные фолды, seed, learning rate и функция потерь не меняются.

Проверяются два параметра CatBoost:

1. `l2_leaf_reg` — L2-штраф на значения в листьях;
2. `random_strength` — величина случайного шума при выборе разбиений.

Подбор выполняется последовательно, по одному параметру. Победитель screening подтверждается на всех четырёх временных holdout.

## 0. Режим запуска

Три флага позволяют воспроизводить этапы независимо. При выключенном флаге ноутбук загружает уже сохранённый результат соответствующего этапа. После завершения эксперимента все флаги нужно выключить.

In [1]:
RUN_L2_SCREENING = False
RUN_RANDOM_STRENGTH_SCREENING = False
RUN_FULL_VALIDATION = False

SCREENING_ANCHORS = ('2025-10-22', '2026-01-14')

## 1. Импорты и подготовка данных

Признаки не строятся заново. Загружаются те же готовые v2-срезы, которые использовались для проверки глубины. Это гарантирует сопоставимость результатов.

In [2]:
from __future__ import annotations

from functools import partial
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / 'src').is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import (
    CATBOOST_REGULARIZATION_DIR,
    CATBOOST_TUNING_DIR,
    CATBOOST_V2_SNAPSHOT_DIR,
    VALIDATION_ANCHORS,
    ensure_output_dirs,
)
from src.experiments import run_temporal_experiment
from src.features import load_snapshots
from src.models import make_validation_model
from src.validation import feature_columns, make_temporal_folds, rmsle

ensure_output_dirs()
snapshots = load_snapshots(CATBOOST_V2_SNAPSHOT_DIR, kind='train')
historical_anchors = sorted(snapshots)
features = feature_columns(snapshots[historical_anchors[0]])
assert len(features) == 216

all_folds = make_temporal_folds(historical_anchors, VALIDATION_ANCHORS)
screening_folds = [
    fold for fold in all_folds
    if fold.validation_anchor.isoformat() in SCREENING_ANCHORS
]
print(f'Признаков: {len(features)}')
print('Screening holdout:', [fold.validation_anchor for fold in screening_folds])

Признаков: 216
Screening holdout: [datetime.date(2025, 10, 22), datetime.date(2026, 1, 14)]


## 2. Зафиксированная отправная точка

Текущая модель использует `depth=6`, `l2_leaf_reg=10` и `random_strength=0.5`. Её не переобучаем: метрики и OOF-предсказания уже сохранены после полной проверки в предыдущем ноутбуке.

In [3]:
DEPTH = 6
REFERENCE_L2 = 10.0
REFERENCE_RANDOM_STRENGTH = 0.5

depth6_metrics = pd.read_csv(
    CATBOOST_TUNING_DIR / 'depth_winner_multifold_metrics.csv'
)
depth6_oof = pd.read_parquet(
    CATBOOST_TUNING_DIR / 'depth_winner_oof.parquet'
)
reference_oof_rmsle = rmsle(
    depth6_oof['target'].to_numpy(),
    depth6_oof['prediction'].to_numpy(),
)
print(f'Отправная точка depth=6: global OOF RMSLE={reference_oof_rmsle:.6f}')

Отправная точка depth=6: global OOF RMSLE=1.731068


## 3. Screening параметра l2_leaf_reg

Меньший штраф (`3`) разрешает более выраженные значения в листьях и потенциально лучше подгоняет сложные зависимости. Больший штраф (`30`) сильнее сглаживает прогноз. Значение `10` является текущей отправной точкой и берётся из сохранённых метрик.

In [4]:
L2_CANDIDATES = {
    'l2_3': 3.0,
    'l2_30': 30.0,
}
l2_screening_path = CATBOOST_REGULARIZATION_DIR / 'l2_screening_metrics.csv'

if RUN_L2_SCREENING:
    l2_parts = []
    for candidate_name, l2_value in L2_CANDIDATES.items():
        metrics, _ = run_temporal_experiment(
            snapshots=snapshots,
            folds=screening_folds,
            features=features,
            model_factory=partial(
                make_validation_model,
                depth=DEPTH,
                l2_leaf_reg=l2_value,
                random_strength=REFERENCE_RANDOM_STRENGTH,
            ),
            keep_oof=False,
            label=candidate_name,
        )
        l2_parts.append(metrics)
    l2_new_metrics = pd.concat(l2_parts, ignore_index=True)
    l2_new_metrics.to_csv(l2_screening_path, index=False)
else:
    if not l2_screening_path.exists():
        raise FileNotFoundError('Нет результатов l2 screening.')
    l2_new_metrics = pd.read_csv(l2_screening_path)

l2_reference = depth6_metrics[
    depth6_metrics['validation_anchor'].isin(SCREENING_ANCHORS)
].copy()
l2_reference['experiment'] = 'l2_10_reference'
l2_comparison = pd.concat(
    [l2_new_metrics, l2_reference[l2_new_metrics.columns]],
    ignore_index=True,
)
display(l2_comparison.sort_values(['validation_anchor', 'catboost_rmsle']))

[l2_3] holdout 2025-10-22


0:	learn: 2.2815045	test: 2.2958255	best: 2.2958255 (0)	total: 558ms	remaining: 16m 43s


200:	learn: 1.6935548	test: 1.7170801	best: 1.7170801 (200)	total: 1m 41s	remaining: 13m 25s


400:	learn: 1.6888459	test: 1.7153662	best: 1.7153662 (400)	total: 3m 3s	remaining: 10m 41s


600:	learn: 1.6851611	test: 1.7149044	best: 1.7149044 (600)	total: 4m 24s	remaining: 8m 46s


...

600:	learn: 1.7037613	test: 1.7074568	best: 1.7074542 (560)	total: 4m 43s	remaining: 9m 26s


800:	learn: 1.7022679	test: 1.7070265	best: 1.7068962 (734)	total: 6m 10s	remaining: 7m 42s


Stopped by overfitting detector  (200 iterations wait)

bestTest = 1.706896162
bestIteration = 734

Shrink model to first 735 iterations.


  RMSLE=1.706892; baseline=2.195065


,experiment,validation_anchor,n_features,n_train_rows,catboost_rmsle,baseline_rmsle,improvement,best_iteration
2,l2_30,2025-10-22,216,750000,1.714563,2.158052,0.443489,991
0,l2_3,2025-10-22,216,750000,1.714598,2.158052,0.443454,1086
4,l2_10_reference,2025-10-22,216,750000,1.714746,2.158052,0.443306,1302
5,l2_10_reference,2026-01-14,216,1500000,1.703598,2.195065,0.491467,722
1,l2_3,2026-01-14,216,1500000,1.705974,2.195065,0.489091,888
3,l2_30,2026-01-14,216,1500000,1.706892,2.195065,0.488172,734


## 4. Выбор l2_leaf_reg

Основной screening-критерий — средний RMSLE раннего и позднего фолдов. Худший из двух RMSLE используется как дополнительная проверка устойчивости.

In [5]:
l2_summary = (
    l2_comparison.groupby('experiment', as_index=False)
    .agg(
        mean_rmsle=('catboost_rmsle', 'mean'),
        worst_rmsle=('catboost_rmsle', 'max'),
        mean_best_iteration=('best_iteration', 'mean'),
    )
    .sort_values(['mean_rmsle', 'worst_rmsle'])
)
display(l2_summary)

l2_winner_name = l2_summary.iloc[0]['experiment']
selected_l2 = (
    REFERENCE_L2
    if l2_winner_name == 'l2_10_reference'
    else L2_CANDIDATES[l2_winner_name]
)
print(f'Победитель l2 screening: {l2_winner_name}; selected_l2={selected_l2}')

,experiment,mean_rmsle,worst_rmsle,mean_best_iteration
0,l2_10_reference,1.709172,1.714746,1012.0
1,l2_3,1.710286,1.714598,987.0
2,l2_30,1.710728,1.714563,862.5


Победитель l2 screening: l2_10_reference; selected_l2=10.0


## 5. Screening параметра random_strength

Теперь фиксируем выбранный `l2_leaf_reg`. Значение `0` полностью убирает шум при оценке разбиений; `1` добавляет больше случайной регуляризации, чем текущие `0.5`. Текущие `0.5` представлены метриками победителя предыдущего этапа и повторно не обучаются.

In [6]:
RANDOM_STRENGTH_CANDIDATES = {
    'random_strength_0': 0.0,
    'random_strength_1': 1.0,
}
random_screening_path = (
    CATBOOST_REGULARIZATION_DIR / 'random_strength_screening_metrics.csv'
)

if RUN_RANDOM_STRENGTH_SCREENING:
    random_parts = []
    for candidate_name, random_value in RANDOM_STRENGTH_CANDIDATES.items():
        label = f'{candidate_name}_l2_{selected_l2:g}'
        metrics, _ = run_temporal_experiment(
            snapshots=snapshots,
            folds=screening_folds,
            features=features,
            model_factory=partial(
                make_validation_model,
                depth=DEPTH,
                l2_leaf_reg=selected_l2,
                random_strength=random_value,
            ),
            keep_oof=False,
            label=label,
        )
        metrics['random_strength'] = random_value
        metrics['l2_leaf_reg'] = selected_l2
        random_parts.append(metrics)
    random_new_metrics = pd.concat(random_parts, ignore_index=True)
    random_new_metrics.to_csv(random_screening_path, index=False)
else:
    if not random_screening_path.exists():
        raise FileNotFoundError('Нет результатов random_strength screening.')
    random_new_metrics = pd.read_csv(random_screening_path)

if selected_l2 == REFERENCE_L2:
    random_reference = l2_reference.copy()
else:
    random_reference = l2_new_metrics[
        l2_new_metrics['experiment'] == l2_winner_name
    ].copy()
random_reference['experiment'] = f'random_strength_0.5_l2_{selected_l2:g}'
random_reference['random_strength'] = REFERENCE_RANDOM_STRENGTH
random_reference['l2_leaf_reg'] = selected_l2

random_comparison = pd.concat(
    [random_new_metrics, random_reference[random_new_metrics.columns]],
    ignore_index=True,
)
display(random_comparison.sort_values(['validation_anchor', 'catboost_rmsle']))

[random_strength_0_l2_10] holdout 2025-10-22


0:	learn: 2.2813276	test: 2.2956834	best: 2.2956834 (0)	total: 357ms	remaining: 10m 42s


200:	learn: 1.6925522	test: 1.7166641	best: 1.7166641 (200)	total: 56s	remaining: 7m 25s


400:	learn: 1.6881636	test: 1.7153253	best: 1.7153253 (400)	total: 1m 44s	remaining: 6m 5s


600:	learn: 1.6848826	test: 1.7149195	best: 1.7149143 (599)	total: 2m 50s	remaining: 5m 41s


...

1200:	learn: 1.6987963	test: 1.7058959	best: 1.7058806 (1175)	total: 9m 59s	remaining: 4m 59s


1400:	learn: 1.6973460	test: 1.7058962	best: 1.7058254 (1355)	total: 11m 30s	remaining: 3m 16s


Stopped by overfitting detector  (200 iterations wait)

bestTest = 1.705825408
bestIteration = 1355

Shrink model to first 1356 iterations.


  RMSLE=1.705820; baseline=2.195065


,experiment,validation_anchor,n_features,n_train_rows,catboost_rmsle,baseline_rmsle,improvement,best_iteration,random_strength,l2_leaf_reg
0,random_strength_0_l2_10,2025-10-22,216,750000,1.714578,2.158052,0.443474,1173,0.0,10.0
2,random_strength_1_l2_10,2025-10-22,216,750000,1.714648,2.158052,0.443404,1092,1.0,10.0
4,random_strength_0.5_l2_10,2025-10-22,216,750000,1.714746,2.158052,0.443306,1302,0.5,10.0
5,random_strength_0.5_l2_10,2026-01-14,216,1500000,1.703598,2.195065,0.491467,722,0.5,10.0
3,random_strength_1_l2_10,2026-01-14,216,1500000,1.705820,2.195065,0.489245,1355,1.0,10.0
1,random_strength_0_l2_10,2026-01-14,216,1500000,1.707670,2.195065,0.487394,1652,0.0,10.0


## 6. Выбор random_strength

In [7]:
random_summary = (
    random_comparison.groupby(
        ['experiment', 'random_strength', 'l2_leaf_reg'], as_index=False
    )
    .agg(
        mean_rmsle=('catboost_rmsle', 'mean'),
        worst_rmsle=('catboost_rmsle', 'max'),
        mean_best_iteration=('best_iteration', 'mean'),
    )
    .sort_values(['mean_rmsle', 'worst_rmsle'])
)
display(random_summary)

regularization_winner = random_summary.iloc[0]
selected_random_strength = float(regularization_winner['random_strength'])
selected_l2 = float(regularization_winner['l2_leaf_reg'])
winner_label = (
    f'depth_6_l2_{selected_l2:g}_random_strength_{selected_random_strength:g}'
)
print('Победитель regularization screening:', winner_label)

,experiment,random_strength,l2_leaf_reg,mean_rmsle,worst_rmsle,mean_best_iteration
0,random_strength_0.5_l2_10,0.5,10.0,1.709172,1.714746,1012.0
2,random_strength_1_l2_10,1.0,10.0,1.710234,1.714648,1223.5
1,random_strength_0_l2_10,0.0,10.0,1.711124,1.714578,1412.5


Победитель regularization screening: depth_6_l2_10_random_strength_0.5


## 7. Полная четырёхфолдовая проверка

Итоговая комбинация screening обучается на всех четырёх временных фолдах. Если screening оставил исходные `l2=10` и `random_strength=0.5`, используются уже рассчитанные OOF-предсказания `depth=6`.

In [8]:
full_metrics_path = CATBOOST_REGULARIZATION_DIR / 'winner_multifold_metrics.csv'
full_oof_path = CATBOOST_REGULARIZATION_DIR / 'winner_oof.parquet'
is_reference = (
    selected_l2 == REFERENCE_L2
    and selected_random_strength == REFERENCE_RANDOM_STRENGTH
)

if RUN_FULL_VALIDATION and not is_reference:
    full_metrics, full_oof = run_temporal_experiment(
        snapshots=snapshots,
        folds=all_folds,
        features=features,
        model_factory=partial(
            make_validation_model,
            depth=DEPTH,
            l2_leaf_reg=selected_l2,
            random_strength=selected_random_strength,
        ),
        keep_oof=True,
        label=winner_label,
    )
    full_metrics.to_csv(full_metrics_path, index=False)
    full_oof.to_parquet(full_oof_path, index=False)
elif is_reference:
    full_metrics = depth6_metrics.copy()
    full_oof = depth6_oof.copy()
elif full_metrics_path.exists() and full_oof_path.exists():
    full_metrics = pd.read_csv(full_metrics_path)
    full_oof = pd.read_parquet(full_oof_path)
else:
    full_metrics = None
    full_oof = None
    print('Полная проверка ещё не выполнена.')

## 8. Сравнение с depth=6

Решение принимается по global OOF RMSLE, а таблица по отдельным фолдам показывает, является ли улучшение устойчивым или возникло только на одной дате.

In [9]:
if full_metrics is not None:
    fold_comparison = depth6_metrics[[
        'validation_anchor', 'catboost_rmsle'
    ]].rename(columns={'catboost_rmsle': 'depth6_reference_rmsle'}).merge(
        full_metrics[['validation_anchor', 'catboost_rmsle', 'best_iteration']],
        on='validation_anchor',
        how='inner',
    ).rename(columns={'catboost_rmsle': 'regularized_rmsle'})
    fold_comparison['improvement'] = (
        fold_comparison['depth6_reference_rmsle']
        - fold_comparison['regularized_rmsle']
    )
    display(fold_comparison)

    winner_oof_rmsle = rmsle(
        full_oof['target'].to_numpy(),
        full_oof['prediction'].to_numpy(),
    )
    global_improvement = reference_oof_rmsle - winner_oof_rmsle
    print(f'Параметры: depth=6, l2={selected_l2:g}, random_strength={selected_random_strength:g}')
    print(f'Reference global OOF RMSLE: {reference_oof_rmsle:.6f}')
    print(f'Winner global OOF RMSLE:    {winner_oof_rmsle:.6f}')
    print(f'Улучшение:                  {global_improvement:+.6f}')

,validation_anchor,depth6_reference_rmsle,regularized_rmsle,best_iteration,improvement
0,2025-10-22,1.714746,1.714746,1302,0.0
1,2025-11-19,1.752591,1.752591,1315,0.0
2,2025-12-17,1.752772,1.752772,1234,0.0
3,2026-01-14,1.703598,1.703598,722,0.0


Параметры: depth=6, l2=10, random_strength=0.5
Reference global OOF RMSLE: 1.731068
Winner global OOF RMSLE:    1.731068
Улучшение:                  +0.000000


## 9. Итог эксперимента

### l2_leaf_reg

| Значение | RMSLE 22.10 | RMSLE 14.01 | Средний RMSLE |
|---:|---:|---:|---:|
| 10 | 1.714746 | 1.703598 | **1.709172** |
| 3 | 1.714598 | 1.705974 | 1.710286 |
| 30 | 1.714563 | 1.706892 | 1.710728 |

Значения `3` и `30` немного улучшили октябрьский фолд, но сильнее ухудшили январский. Исходное `l2_leaf_reg=10` сохранилось.

### random_strength

| Значение | RMSLE 22.10 | RMSLE 14.01 | Средний RMSLE |
|---:|---:|---:|---:|
| 0.5 | 1.714746 | 1.703598 | **1.709172** |
| 1.0 | 1.714648 | 1.705820 | 1.710234 |
| 0.0 | 1.714578 | 1.707670 | 1.711124 |

Значения `0` и `1` также немного улучшили октябрьский фолд, но проиграли на январском. Исходное `random_strength=0.5` сохранилось. При `random_strength=0` январская модель дошла до 1652-й итерации, то есть слабый результат не объясняется слишком ранним early stopping.

Итоговая рабочая конфигурация не изменилась: `depth=6`, `l2_leaf_reg=10`, `random_strength=0.5`. Её global OOF RMSLE остаётся `1.731068`. Дополнительная полная валидация не запускалась, потому что оба screening оставили уже проверенную на четырёх фолдах reference-конфигурацию.

Новый submission в данном ноутбуке не создаётся. Следующий независимый эксперимент — подбор доли признаков `rsm`. Все флаги выше выключены: обычный перезапуск загружает сохранённые результаты без обучения.